# Imports

In [2]:
import xarray as xr
from dask_jobqueue import PBSCluster
from dask.distributed import Client

# PBSCluster
Kernel keeps crashing due to resolution

In [3]:
cluster = PBSCluster(
    cores=1, # The number of cores you want
    memory='16GB', # Amount of memory
    processes=1, # How many processes
    queue='casper', # The type of queue to utilize (/glade/u/apps/dav/opt/usr/bin/execcasper)
    local_directory='$TMPDIR', # Use your local directory
    resource_spec='select=1:ncpus=1:mem=16GB', # Specify resources
    account='P93300313', # Input your project ID here
    walltime='01:00:00', # Amount of wall time
    n_workers=4,
)
# Setup your client
client = Client(cluster)

In [4]:
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.182:32959,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


# Load

In [5]:
ds = xr.open_dataset('/glade/work/acruz/Reanalysis/ERA5_UW.nc', chunks='auto')
ds.nbytes

162264477040

In [3]:
ds

<xarray.Dataset> Size: 162GB
Dimensions:    (time: 528, level: 37, latitude: 721, longitude: 1440)
Coordinates:
  * latitude   (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * level      (level) float64 296B 1.0 2.0 3.0 5.0 ... 925.0 950.0 975.0 1e+03
  * longitude  (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
  * time       (time) datetime64[ns] 4kB 1979-01-01 1979-02-01 ... 2022-12-01
Data variables:
    U          (time, level, latitude, longitude) float32 81GB ...
    utc_date   (time) int32 2kB ...
    W          (time, level, latitude, longitude) float32 81GB ...
Attributes:
    DATA_SOURCE:          ECMWF: https://cds.climate.copernicus.eu, Copernicu...
    NETCDF_CONVERSION:    CISL RDA: Conversion from ECMWF GRIB 1 data to netC...
    NETCDF_VERSION:       4.6.1
    CONVERSION_PLATFORM:  Linux casper04 3.10.0-693.21.1.el7.x86_64 #1 SMP We...
    CONVERSION_DATE:      Fri Nov  8 18:03:42 MST 2019
    Conventions:          CF-1.6
    NETCDF_COMPRESSION:   NCO: Precision-preserving compression to netCDF4/HD...
    history:              Fri Nov  8 18:03:52 2019: ncks -4 --ppc default=7 e...
    NCO:                  netCDF Operators version 4.7.9 (Homepage = http://n...

# Select Data

In [6]:
ds.coords['longitude'] = (ds.coords['longitude'] + 180) % 360 - 180
ds = ds.sortby(ds.longitude)

In [7]:
# making sure the latitude transformation works properly
newlats = (ds.coords['latitude'] + 90) % 180 - 90
newlats.sortby(newlats.latitude)

<xarray.DataArray 'latitude' (latitude: 721)> Size: 6kB
array([-90.  , -89.75, -89.5 , -89.25, -89.  , -88.75, -88.5 , -88.25,
       -88.  , -87.75, -87.5 , -87.25, -87.  , -86.75, -86.5 , -86.25,
       -86.  , -85.75, -85.5 , -85.25, -85.  , -84.75, -84.5 , -84.25,
       -84.  , -83.75, -83.5 , -83.25, -83.  , -82.75, -82.5 , -82.25,
       -82.  , -81.75, -81.5 , -81.25, -81.  , -80.75, -80.5 , -80.25,
       -80.  , -79.75, -79.5 , -79.25, -79.  , -78.75, -78.5 , -78.25,
       -78.  , -77.75, -77.5 , -77.25, -77.  , -76.75, -76.5 , -76.25,
       -76.  , -75.75, -75.5 , -75.25, -75.  , -74.75, -74.5 , -74.25,
       -74.  , -73.75, -73.5 , -73.25, -73.  , -72.75, -72.5 , -72.25,
       -72.  , -71.75, -71.5 , -71.25, -71.  , -70.75, -70.5 , -70.25,
       -70.  , -69.75, -69.5 , -69.25, -69.  , -68.75, -68.5 , -68.25,
       -68.  , -67.75, -67.5 , -67.25, -67.  , -66.75, -66.5 , -66.25,
       -66.  , -65.75, -65.5 , -65.25, -65.  , -64.75, -64.5 , -64.25,
       -64.  , -63.75, -63.5 , -63.25, -63.  , -62.75, -62.5 , -62.25,
       -62.  , -61.75, -61.5 , -61.25, -61.  , -60.75, -60.5 , -60.25,
       -60.  , -59.75, -59.5 , -59.25, -59.  , -58.75, -58.5 , -58.25,
       -58.  , -57.75, -57.5 , -57.25, -57.  , -56.75, -56.5 , -56.25,
       -56.  , -55.75, -55.5 , -55.25, -55.  , -54.75, -54.5 , -54.25,
       -54.  , -53.75, -53.5 , -53.25, -53.  , -52.75, -52.5 , -52.25,
       -52.  , -51.75, -51.5 , -51.25, -51.  , -50.75, -50.5 , -50.25,
...
        52.  ,  52.25,  52.5 ,  52.75,  53.  ,  53.25,  53.5 ,  53.75,
        54.  ,  54.25,  54.5 ,  54.75,  55.  ,  55.25,  55.5 ,  55.75,
        56.  ,  56.25,  56.5 ,  56.75,  57.  ,  57.25,  57.5 ,  57.75,
        58.  ,  58.25,  58.5 ,  58.75,  59.  ,  59.25,  59.5 ,  59.75,
        60.  ,  60.25,  60.5 ,  60.75,  61.  ,  61.25,  61.5 ,  61.75,
        62.  ,  62.25,  62.5 ,  62.75,  63.  ,  63.25,  63.5 ,  63.75,
        64.  ,  64.25,  64.5 ,  64.75,  65.  ,  65.25,  65.5 ,  65.75,
        66.  ,  66.25,  66.5 ,  66.75,  67.  ,  67.25,  67.5 ,  67.75,
        68.  ,  68.25,  68.5 ,  68.75,  69.  ,  69.25,  69.5 ,  69.75,
        70.  ,  70.25,  70.5 ,  70.75,  71.  ,  71.25,  71.5 ,  71.75,
        72.  ,  72.25,  72.5 ,  72.75,  73.  ,  73.25,  73.5 ,  73.75,
        74.  ,  74.25,  74.5 ,  74.75,  75.  ,  75.25,  75.5 ,  75.75,
        76.  ,  76.25,  76.5 ,  76.75,  77.  ,  77.25,  77.5 ,  77.75,
        78.  ,  78.25,  78.5 ,  78.75,  79.  ,  79.25,  79.5 ,  79.75,
        80.  ,  80.25,  80.5 ,  80.75,  81.  ,  81.25,  81.5 ,  81.75,
        82.  ,  82.25,  82.5 ,  82.75,  83.  ,  83.25,  83.5 ,  83.75,
        84.  ,  84.25,  84.5 ,  84.75,  85.  ,  85.25,  85.5 ,  85.75,
        86.  ,  86.25,  86.5 ,  86.75,  87.  ,  87.25,  87.5 ,  87.75,
        88.  ,  88.25,  88.5 ,  88.75,  89.  ,  89.25,  89.5 ,  89.75,
       -90.  ])
Coordinates:
  * latitude  (latitude) float64 6kB -90.0 -89.75 -89.5 ... 89.5 89.75 90.0

In [8]:
ds = ds.sortby(newlats.latitude)
ds

<xarray.Dataset> Size: 162GB
Dimensions:    (time: 528, level: 37, latitude: 721, longitude: 1440)
Coordinates:
  * latitude   (latitude) float64 6kB -90.0 -89.75 -89.5 ... 89.5 89.75 90.0
  * level      (level) float64 296B 1.0 2.0 3.0 5.0 ... 925.0 950.0 975.0 1e+03
  * time       (time) datetime64[ns] 4kB 1979-01-01 1979-02-01 ... 2022-12-01
  * longitude  (longitude) float64 12kB -180.0 -179.8 -179.5 ... 179.5 179.8
Data variables:
    U          (time, level, latitude, longitude) float32 81GB dask.array<chunksize=(59, 3, 73, 80), meta=np.ndarray>
    utc_date   (time) int32 2kB dask.array<chunksize=(528,), meta=np.ndarray>
    W          (time, level, latitude, longitude) float32 81GB dask.array<chunksize=(59, 3, 73, 80), meta=np.ndarray>
Attributes:
    DATA_SOURCE:          ECMWF: https://cds.climate.copernicus.eu, Copernicu...
    NETCDF_CONVERSION:    CISL RDA: Conversion from ECMWF GRIB 1 data to netC...
    NETCDF_VERSION:       4.6.1
    CONVERSION_PLATFORM:  Linux casper04 3.10.0-693.21.1.el7.x86_64 #1 SMP We...
    CONVERSION_DATE:      Fri Nov  8 18:03:42 MST 2019
    Conventions:          CF-1.6
    NETCDF_COMPRESSION:   NCO: Precision-preserving compression to netCDF4/HD...
    history:              Fri Nov  8 18:03:52 2019: ncks -4 --ppc default=7 e...
    NCO:                  netCDF Operators version 4.7.9 (Homepage = http://n...

In [9]:
tropic = ds.sel(latitude=slice(-10, 10), longitude=slice(-180, 20))
tropic

<xarray.Dataset> Size: 10GB
Dimensions:    (time: 528, level: 37, latitude: 81, longitude: 801)
Coordinates:
  * latitude   (latitude) float64 648B -10.0 -9.75 -9.5 -9.25 ... 9.5 9.75 10.0
  * level      (level) float64 296B 1.0 2.0 3.0 5.0 ... 925.0 950.0 975.0 1e+03
  * time       (time) datetime64[ns] 4kB 1979-01-01 1979-02-01 ... 2022-12-01
  * longitude  (longitude) float64 6kB -180.0 -179.8 -179.5 ... 19.5 19.75 20.0
Data variables:
    U          (time, level, latitude, longitude) float32 5GB dask.array<chunksize=(59, 3, 77, 80), meta=np.ndarray>
    utc_date   (time) int32 2kB dask.array<chunksize=(528,), meta=np.ndarray>
    W          (time, level, latitude, longitude) float32 5GB dask.array<chunksize=(59, 3, 77, 80), meta=np.ndarray>
Attributes:
    DATA_SOURCE:          ECMWF: https://cds.climate.copernicus.eu, Copernicu...
    NETCDF_CONVERSION:    CISL RDA: Conversion from ECMWF GRIB 1 data to netC...
    NETCDF_VERSION:       4.6.1
    CONVERSION_PLATFORM:  Linux casper04 3.10.0-693.21.1.el7.x86_64 #1 SMP We...
    CONVERSION_DATE:      Fri Nov  8 18:03:42 MST 2019
    Conventions:          CF-1.6
    NETCDF_COMPRESSION:   NCO: Precision-preserving compression to netCDF4/HD...
    history:              Fri Nov  8 18:03:52 2019: ncks -4 --ppc default=7 e...
    NCO:                  netCDF Operators version 4.7.9 (Homepage = http://n...

# Calculate

In [10]:
tropic.nbytes

10140135416

In [11]:
clim = tropic.groupby('time.month').mean(dim='time').compute()
clim

<xarray.Dataset> Size: 230MB
Dimensions:    (month: 12, level: 37, latitude: 81, longitude: 801)
Coordinates:
  * latitude   (latitude) float64 648B -10.0 -9.75 -9.5 -9.25 ... 9.5 9.75 10.0
  * level      (level) float64 296B 1.0 2.0 3.0 5.0 ... 925.0 950.0 975.0 1e+03
  * longitude  (longitude) float64 6kB -180.0 -179.8 -179.5 ... 19.5 19.75 20.0
  * month      (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
Data variables:
    U          (month, level, latitude, longitude) float32 115MB -61.65 ... -...
    utc_date   (month) float64 96B 2.001e+09 2.001e+09 ... 2.001e+09 2.001e+09
    W          (month, level, latitude, longitude) float32 115MB nan nan ... nan
Attributes:
    DATA_SOURCE:          ECMWF: https://cds.climate.copernicus.eu, Copernicu...
    NETCDF_CONVERSION:    CISL RDA: Conversion from ECMWF GRIB 1 data to netC...
    NETCDF_VERSION:       4.6.1
    CONVERSION_PLATFORM:  Linux casper04 3.10.0-693.21.1.el7.x86_64 #1 SMP We...
    CONVERSION_DATE:      Fri Nov  8 18:03:42 MST 2019
    Conventions:          CF-1.6
    NETCDF_COMPRESSION:   NCO: Precision-preserving compression to netCDF4/HD...
    history:              Fri Nov  8 18:03:52 2019: ncks -4 --ppc default=7 e...
    NCO:                  netCDF Operators version 4.7.9 (Homepage = http://n...

In [17]:
u_tropic, u_clim = xr.unify_chunks(tropic, clim)

In [18]:
tropic_anom = u_tropic.groupby('time.month') - u_clim
tropic_anom

/glade/u/home/acruz/.local/lib/python3.10/site-packages/xarray/core/indexing.py:1680: PerformanceWarning: Slicing with an out-of-order index is generating 44 times more chunks
  return self.array[key]


<xarray.Dataset> Size: 10GB
Dimensions:    (latitude: 81, level: 37, time: 528, longitude: 801)
Coordinates:
  * latitude   (latitude) float64 648B -10.0 -9.75 -9.5 -9.25 ... 9.5 9.75 10.0
  * level      (level) float64 296B 1.0 2.0 3.0 5.0 ... 925.0 950.0 975.0 1e+03
  * time       (time) datetime64[ns] 4kB 1979-01-01 1979-02-01 ... 2022-12-01
  * longitude  (longitude) float64 6kB -180.0 -179.8 -179.5 ... 19.5 19.75 20.0
    month      (time) int64 4kB 1 2 3 4 5 6 7 8 9 10 ... 3 4 5 6 7 8 9 10 11 12
Data variables:
    U          (time, level, latitude, longitude) float32 5GB dask.array<chunksize=(1, 3, 77, 80), meta=np.ndarray>
    utc_date   (time) float64 4kB dask.array<chunksize=(1,), meta=np.ndarray>
    W          (time, level, latitude, longitude) float32 5GB dask.array<chunksize=(1, 3, 77, 80), meta=np.ndarray>

In [19]:
tropic_anom.compute()

/glade/u/home/acruz/.conda/envs/EOFa_2025/lib/python3.10/site-packages/distributed/client.py:3149: UserWarning: Sending large graph of size 260.18 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


<xarray.Dataset> Size: 10GB
Dimensions:    (latitude: 81, level: 37, time: 528, longitude: 801)
Coordinates:
  * latitude   (latitude) float64 648B -10.0 -9.75 -9.5 -9.25 ... 9.5 9.75 10.0
  * level      (level) float64 296B 1.0 2.0 3.0 5.0 ... 925.0 950.0 975.0 1e+03
  * time       (time) datetime64[ns] 4kB 1979-01-01 1979-02-01 ... 2022-12-01
  * longitude  (longitude) float64 6kB -180.0 -179.8 -179.5 ... 19.5 19.75 20.0
    month      (time) int64 4kB 1 2 3 4 5 6 7 8 9 10 ... 3 4 5 6 7 8 9 10 11 12
Data variables:
    U          (time, level, latitude, longitude) float32 5GB nan nan ... nan
    utc_date   (time) float64 4kB -2.15e+07 -2.15e+07 ... 2.15e+07 2.15e+07
    W          (time, level, latitude, longitude) float32 5GB nan nan ... nan

In [ ]:
tropic_anom.to_netcdf('/glade/work/acruz/Reanalysis/ERA5_UW_tropic_anom.nc')

/glade/u/home/acruz/.conda/envs/EOFa_2025/lib/python3.10/site-packages/distributed/client.py:3149: UserWarning: Sending large graph of size 274.81 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


In [ ]:
tropic_anom.nbytes